# Capstone: casas_analysis

Describes the CASAS data (the followed article's) for the dataset analysis: when the common activities
start, how often, and how often an activity repeats from one sensor interval to the next. Downloads
CASAS (236 MB) and prepares it, then writes a small summary file.

**Settings:** Accelerator = **None** (CPU), Internet = **On**, dataset `capstone-bundle` (latest version)
under Input. Save Version > Save & Run All (Commit).

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('casas_analysis: started', 'CASAS download, prepare, summary', 'default', 'rocket')

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/casas.zip "https://zenodo.org/records/15708568/files/labeled_data.zip?download=1"
!ls -l /kaggle/temp/casas.zip   # should be 236037656 bytes
ok = run_step('CASAS prepare', 'CAPSTONE_DATASET=casas python -u scripts/prepare_casas.py /kaggle/temp/casas.zip',
              'logs/casas_prepare.log', summary_lines=3)
casas_ok = ok  # without CASAS, its steps are skipped (REFIT and UK-DALE still run)
if casas_ok:
    ok &= run_step('CASAS summary', 'python -u scripts/analyze_casas.py', 'logs/casas_analysis.log',
                   env={'CAPSTONE_DATASET': 'casas'}, summary_lines=5)

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/casas_analysis.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [*pathlib.Path('data/results').rglob('casas_summary.json'), *pathlib.Path('logs').rglob('*.log')]:
        zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/casas_analysis.zip').stat().st_size / 1e6, 3), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('casas_analysis: FINISHED' if ok else 'casas_analysis: finished WITH ERRORS',
       'casas_analysis.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')